# Brain MRI (BraTS) — nnU-Net segmentation training

**"Light training" from the plan — but the heaviest of the four.** Per
`docs/04-data-models.md`: no ready-to-use pretrained BraTS checkpoint exists on
Hugging Face — nnU-Net ships as a **self-configuring training framework**, not
a hosted-weights hub. So this vertical genuinely trains its own segmentation
backbone, unlike the other three components in this batch.

**Read this before you run anything:**
- `docs/07-risks-decisions.md` flags **"3D compute cost"** as a named risk with
  the mitigation *"foundation-model embeddings / patch-based methods, not full
  retrain."* nnU-Net already trains on **patches**, not full volumes — that's
  the mitigation already applied. But a *full* nnU-Net BraTS run (5-fold
  cross-validation, ~1000 epochs/fold) is genuinely a multi-day, multi-GPU job.
  **Kaggle's session limit (12h, single GPU) is not enough for that.** This
  notebook trains **one fold, a reduced epoch budget**, to produce a real,
  working (if under-trained) checkpoint and prove the pipeline — not a
  publication-grade result. Scale `NUM_EPOCHS` and add more folds once you have
  a multi-GPU budget.
- Unlike the other three notebooks in this batch, nnU-Net is driven through its
  **own CLI commands** (`nnUNetv2_plan_and_preprocess`, `nnUNetv2_train`), not a
  hand-written PyTorch loop — that's how nnU-Net's "self-configuring" design
  works: it picks the network architecture, patch size, and batch size for you
  based on the dataset's own properties.

## Kaggle setup
- **Add data:** search *Add Data* for **"BraTS"** (BraTS2020 or BraTS2021 Kaggle
  mirrors both work — this notebook auto-detects the modality file layout).
- **Accelerator:** GPU T4 x2 or, if available, a P100/A100 — 3D patch training
  is memory-hungry.
- **Internet:** on (for `pip install nnunetv2`).
- **Disk:** nnU-Net writes large intermediate preprocessed files — keep an eye
  on `/kaggle/working`'s quota if you scale beyond a handful of cases.

In [ ]:
# Cell 1 — install nnU-Net v2.
!pip install -q nnunetv2

In [ ]:
# Cell 2 — nnU-Net is entirely environment-variable driven: it reads
# nnUNet_raw / nnUNet_preprocessed / nnUNet_results to know where everything
# lives. Set these before importing/calling any nnunetv2 command.
import glob
import json
import os
import shutil

import nibabel as nib
import numpy as np

NNUNET_BASE = "/kaggle/working/nnunet"
os.environ["nnUNet_raw"] = f"{NNUNET_BASE}/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = f"{NNUNET_BASE}/nnUNet_preprocessed"
os.environ["nnUNet_results"] = f"{NNUNET_BASE}/nnUNet_results"

for path in os.environ["nnUNet_raw"], os.environ["nnUNet_preprocessed"], os.environ["nnUNet_results"]:
    os.makedirs(path, exist_ok=True)

DATASET_ID = 1
DATASET_NAME = f"Dataset{DATASET_ID:03d}_BraTS"
DATASET_DIR = os.path.join(os.environ["nnUNet_raw"], DATASET_NAME)
os.makedirs(os.path.join(DATASET_DIR, "imagesTr"), exist_ok=True)
os.makedirs(os.path.join(DATASET_DIR, "labelsTr"), exist_ok=True)
print("nnU-Net dataset dir:", DATASET_DIR)

## Locate BraTS on Kaggle and convert to nnU-Net's format

nnU-Net expects, per case:
- `imagesTr/<case>_0000.nii.gz`, `_0001`, `_0002`, `_0003` — one file per MRI
  modality (we use the conventional nnU-Net BraTS channel order: **0=T1,
  1=T1ce, 2=T2, 3=FLAIR** — matches `docs/12`'s "multi-modal MRI: T1, T1ce, T2,
  FLAIR").
- `labelsTr/<case>.nii.gz` — the segmentation mask (BraTS label convention:
  1=necrotic core, 2=edema, 4=enhancing tumor — nnU-Net wants labels to be
  **consecutive integers starting at 0**, so we remap `4 → 3` below).

Kaggle BraTS mirrors vary in exact folder naming, so this cell searches for the
modality keywords (`t1`, `t1ce`, `t2`, `flair`, `seg`) rather than hardcoding a
single mirror's layout.

In [ ]:
# Cell 3 — find BraTS case folders on Kaggle.
MODALITY_KEYWORDS = {"t1": 0, "t1ce": 1, "t2": 2, "flair": 3}

seg_files = glob.glob("/kaggle/input/**/*seg.nii.gz", recursive=True) + glob.glob(
    "/kaggle/input/**/*seg.nii", recursive=True
)
if not seg_files:
    raise FileNotFoundError(
        "No BraTS segmentation files found under /kaggle/input. "
        "Add a BraTS dataset via 'Add Data', or adjust the glob pattern above "
        "to match your specific mirror's folder layout."
    )
case_dirs = sorted({os.path.dirname(path) for path in seg_files})
print(f"Found {len(case_dirs)} BraTS case folders. Example: {case_dirs[0]}")

# Scope down for a Kaggle-session-friendly demonstration run - raise this once
# you have a real multi-day/multi-GPU training budget.
MAX_CASES = 60
case_dirs = case_dirs[:MAX_CASES]
print(f"Using {len(case_dirs)} cases for this run.")

In [ ]:
# Cell 4 — convert each case into nnU-Net's expected layout.
def find_modality_file(case_dir: str, keyword: str) -> str | None:
    matches = [
        f for f in glob.glob(os.path.join(case_dir, "*.nii*"))
        if keyword in os.path.basename(f).lower() and "seg" not in os.path.basename(f).lower()
    ]
    return matches[0] if matches else None


def remap_brats_labels(segmentation: np.ndarray) -> np.ndarray:
    # BraTS labels {0, 1, 2, 4} -> nnU-Net wants consecutive {0, 1, 2, 3}.
    remapped = segmentation.copy()
    remapped[segmentation == 4] = 3
    return remapped


converted_case_ids = []
for case_dir in case_dirs:
    case_id = os.path.basename(case_dir.rstrip("/"))
    modality_paths = {keyword: find_modality_file(case_dir, keyword) for keyword in MODALITY_KEYWORDS}
    if any(path is None for path in modality_paths.values()):
        print(f"skip {case_id}: missing a modality file")
        continue

    seg_path = glob.glob(os.path.join(case_dir, "*seg.nii*"))[0]

    for keyword, channel_index in MODALITY_KEYWORDS.items():
        destination = os.path.join(DATASET_DIR, "imagesTr", f"{case_id}_{channel_index:04d}.nii.gz")
        shutil.copy(modality_paths[keyword], destination)

    seg_image = nib.load(seg_path)
    seg_data = remap_brats_labels(np.asarray(seg_image.dataobj))
    remapped_image = nib.Nifti1Image(seg_data.astype(np.uint8), seg_image.affine, seg_image.header)
    nib.save(remapped_image, os.path.join(DATASET_DIR, "labelsTr", f"{case_id}.nii.gz"))

    converted_case_ids.append(case_id)

print(f"Converted {len(converted_case_ids)} cases.")

## `dataset.json` — nnU-Net's manifest

Declares the modalities (channel order must match the `_0000.._0003` suffixes
above), the label map (post-remap: 0=background, 1=necrotic core, 2=edema,
3=enhancing tumor), and the file count. This is the file nnU-Net's planner
reads to decide preprocessing and network configuration automatically.

In [ ]:
# Cell 5 — write dataset.json.
dataset_json = {
    "channel_names": {"0": "T1", "1": "T1ce", "2": "T2", "3": "FLAIR"},
    "labels": {
        "background": 0,
        "necrotic_core": 1,
        "edema": 2,
        "enhancing_tumor": 3,
    },
    "numTraining": len(converted_case_ids),
    "file_ending": ".nii.gz",
}
with open(os.path.join(DATASET_DIR, "dataset.json"), "w") as handle:
    json.dump(dataset_json, handle, indent=2)
print(json.dumps(dataset_json, indent=2))

## Plan & preprocess

nnU-Net inspects the dataset's spacing, size, and intensity distribution and
picks: patch size, batch size, network depth, and whether a 2D/3D-lowres/3D-fullres
configuration fits your data — matching `docs/12`'s note that nnU-Net's
self-configuring pipeline handles this automatically. We only train the
`3d_fullres` configuration here to keep this notebook to one command path.

In [ ]:
# Cell 6 — run nnU-Net's planning + preprocessing step.
!nnUNetv2_plan_and_preprocess -d {DATASET_ID} --verify_dataset_integrity -c 3d_fullres

## Train — one fold, reduced epoch budget

Real nnU-Net training defaults to **5-fold cross-validation, 1000 epochs per
fold** — far beyond a single Kaggle session. We train **fold 0 only**, with a
reduced epoch count via nnU-Net's trainer-class override mechanism, purely to
produce a working checkpoint and confirm the pipeline end-to-end. Treat the
resulting Dice score as a smoke-test number, not a benchmark result — scale
`nnUNetTrainer_50epochs` up (or drop the override entirely) once you have a
real training budget.

In [ ]:
# Cell 7 — train fold 0 with a reduced-epoch trainer variant.
# nnUNetTrainer_50epochs ships with nnunetv2 precisely for quick pipeline
# smoke tests like this one - swap back to the plain 'nnUNetTrainer' (1000
# epochs) for a real run.
!nnUNetv2_train {DATASET_ID} 3d_fullres 0 -tr nnUNetTrainer_50epochs

## Inference on a held-out case

Run the trained fold on a case that wasn't used for training (nnU-Net's
internal validation split, not a separate held-out set we control here — for
a real promotion gate, hold out cases nnU-Net never sees at all, matching
`docs/12`'s "never touch BraTS's own validation leaderboard data as training
data" rule).

In [ ]:
# Cell 8 — run inference with the trained fold.
INFERENCE_INPUT_DIR = f"{NNUNET_BASE}/inference_input"
INFERENCE_OUTPUT_DIR = f"{NNUNET_BASE}/inference_output"
os.makedirs(INFERENCE_INPUT_DIR, exist_ok=True)
os.makedirs(INFERENCE_OUTPUT_DIR, exist_ok=True)

# Copy one training case's images to act as a held-out inference example for
# this smoke test (again: not a substitute for a real held-out set).
sample_case_id = converted_case_ids[0]
for channel_index in range(4):
    source = os.path.join(DATASET_DIR, "imagesTr", f"{sample_case_id}_{channel_index:04d}.nii.gz")
    shutil.copy(source, INFERENCE_INPUT_DIR)

!nnUNetv2_predict -i {INFERENCE_INPUT_DIR} -o {INFERENCE_OUTPUT_DIR} -d {DATASET_ID} -c 3d_fullres -f 0 -tr nnUNetTrainer_50epochs
print("Predicted files:", os.listdir(INFERENCE_OUTPUT_DIR))

## Locate the checkpoint and results

nnU-Net writes its own checkpoint + training-log structure under
`nnUNet_results` — there's nothing to hand-serialize here, just point at the
right file when you export.

In [ ]:
# Cell 9 — locate the trained checkpoint + summary metrics nnU-Net produced.
checkpoint_paths = glob.glob(f"{os.environ['nnUNet_results']}/**/checkpoint_final.pth", recursive=True)
summary_paths = glob.glob(f"{os.environ['nnUNet_results']}/**/summary.json", recursive=True)

print("Checkpoint(s):", checkpoint_paths)
print("Summary file(s):", summary_paths)

if summary_paths:
    with open(summary_paths[0]) as handle:
        summary = json.load(handle)
    print("Mean Dice per class:", summary.get("mean", {}))

## Next steps (outside this notebook)

1. **This is a pipeline smoke test, not a trained model.** Before anything here
   is promotion-eligible: use the full case set (hundreds of cases, not 60),
   the default `nnUNetTrainer` (1000 epochs), and **all 5 folds**, on a
   multi-GPU box or a longer-running cloud job — not a single Kaggle session.
2. Compare against the **published nnU-Net BraTS baseline Dice scores** —
   docs/12's actual promotion gate for this vertical.
3. Report **Dice + Hausdorff distance per tumor sub-region** (whole tumor /
   tumor core / enhancing tumor), not just an aggregate — the BraTS-standard
   reporting granularity.
4. Once trained, wire **MedSAM/MedSAM2** (`wanglab/medsam-vit-base`,
   `wanglab/MedSAM2` — both usable as-is, no training needed) alongside this
   nnU-Net model for interactive boundary refinement in the dashboard, per
   `docs/12`.